# FRED Bronze Ingestion
Pulls raw economic time series observations from the FRED API (UNRATE, CPIAUCSL, FEDFUNDS, MORTGAGE30US, GDP) and writes to a Bronze Delta table with no transformations.

### Step 1: API key input widget

In [0]:
dbutils.widgets.text("fred_api_key", "", "FRED API Key")
api_key = dbutils.widgets.get("fred_api_key")

### Step 2: Pull observations from FRED API

In [0]:
import requests
import pandas as pd

series_list = ["UNRATE", "CPIAUCSL", "FEDFUNDS", "MORTGAGE30US", "GDP"]

all_observations = []

for series_id in series_list:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json"
    }
    response = requests.get(url, params=params)
    data = response.json()
    
    for obs in data["observations"]:
        all_observations.append({
            "series_id": series_id,
            "date": obs["date"],
            "value": obs["value"]
        })

df_bronze = pd.DataFrame(all_observations)
print(f"Total records pulled: {len(df_bronze)}")
print(f"Series included: {df_bronze['series_id'].unique()}")
display(df_bronze.head(10))

Total records pulled: 5960
Series included: ['UNRATE' 'CPIAUCSL' 'FEDFUNDS' 'MORTGAGE30US' 'GDP']


series_id,date,value
UNRATE,1948-01-01,3.4
UNRATE,1948-02-01,3.8
UNRATE,1948-03-01,4.0
UNRATE,1948-04-01,3.9
UNRATE,1948-05-01,3.5
UNRATE,1948-06-01,3.6
UNRATE,1948-07-01,3.6
UNRATE,1948-08-01,3.9
UNRATE,1948-09-01,3.8
UNRATE,1948-10-01,3.7


### Step 3: Write raw data to Bronze Delta table

In [0]:
# Convert to Spark DataFrame and write to Bronze Delta table
df_bronze_spark = spark.createDataFrame(df_bronze)

df_bronze_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_fred_economic")

print(f"Bronze table written: {df_bronze_spark.count()} records")

Bronze table written: 5960 records
